In [1]:
import os
import torch
import openapi

In [2]:
access_token = os.getenv("HuggingFaceToken")

In [43]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained("mental/mental-bert-base-uncased", token=access_token)
model = AutoModelForSequenceClassification.from_pretrained("mental/mental-bert-base-uncased", token=access_token)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at mental/mental-bert-base-uncased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [54]:
text = tokenizer("i am happy", return_tensors="pt")

In [55]:
# Run model
with torch.no_grad():
    outputs = model(**text)

# Get predicted class
logits = outputs.logits
predicted_class = torch.argmax(logits, dim=1).item()

In [56]:
outputs

SequenceClassifierOutput(loss=None, logits=tensor([[ 0.1185, -0.3061]]), hidden_states=None, attentions=None)

In [46]:
model.config.id2label

{0: 'LABEL_0', 1: 'LABEL_1'}

In [51]:
predicted_class

0

In [47]:
outputs[0]

tensor([[ 0.1016, -0.2365]])

In [37]:
torch.argmax(outputs[0], dim = 2).item()

RuntimeError: a Tensor with 6 elements cannot be converted to Scalar

In [ ]:
class RiskDetectionAgent:
    def __init__(self):
        self.llm = OpenAI(
            api_key="xxx", # input your api key here
            base_url="https://api.deepinfra.com/v1/openai",
        )
        self.temperature = 0.0
        self.max_iterations = 10

        ################################
        # Your code starts here

        ################################
        self.submission_val_path = 'data/submission_val.csv'
        self.val_path = 'data/val.csv'
        self.submission_test_path = 'data/submission_test.csv'

        REACT_PROMPT="""
        You are an AI assistant using ReAct (Reasoning + Acting).
        Generate valid Python code in a string to solve the task.
        The libraries available to you are:
        Pandas
        Scikit-learn
        Generate only the raw Python code, no Markdown formatting, no backticks, no explanations. Just the code itself.
        If there are comments, begin the line with "#". Follow my next instructions carefully.
        """
        ################################
        # Your code ends here
        ################################

        self.SYSTEM_PROMPT = REACT_PROMPT + """
You need to:
1. Train a machine learning model using `data/train.csv`
2. Generate predictions for validation data in `data/val.csv` and save as `data/submission_val.csv`
3. Generate predictions for test data in `data/test_no_labels.csv` and save as `data/submission_test.csv`

Both submission files should have the following format:
```
PassengerId,Survived
899,0
900,1
901,0
902,1
903,0
904,1
```

IMPORTANT: You must generate BOTH `submission_val.csv` and `submission_test.csv` for your solution to be complete.
"""
        self.messages = [{"role": "system", "content": self.SYSTEM_PROMPT}]

    def query_llm(self):
        response = self.llm.chat.completions.create(
            model="Qwen/Qwen2.5-Coder-32B-Instruct",
            messages=self.messages,
            temperature=self.temperature
        )
        return response.choices[0].message.content

    ################################
    # Your code starts here
    #
    # Implement the run method that manages the conversation with the LLM agent.
    # Your implementation should:
    # 1. Initialize the conversation with the user prompt
    # 2. Delete any existing submission files
    # 3. Run an iterative process where:
    #    - The LLM generates code based on previous feedback
    #    - The code is extracted and executed
    #    - Results are evaluated
    #    - Feedback is provided to the LLM for the next iteration
    # 4. Check for the creation of required submission files
    # 5. Validate performance against the accuracy threshold (0.75)
    # 6. Exit the loop if the maximum iteration limit is reached or validation accuracy surpasses the threshold
    ################################
    def run(self, user_prompt):
        try:
            os.remove(self.submission_val_path)
            os.remove(self.submission_test_path)
        except Exception as e:
            pass

        self.messages.append({"role": "user", "content": user_prompt})
        # Add your implementation here
        improvement = 0
        acc = None

        for i in range(self.max_iterations):
            code = self.query_llm()
            test_code_output = code_interpreter(code)
            # Send error back
            if 'returncode' not in test_code_output.keys():
                self.messages.append({"role": "user", "content": "The code encountered a timeout exception. Please fix it"})
                self.messages.append({"role": "assistant", "content": code})
                continue
            
            # Send error back
            if test_code_output['returncode'] != 0:
                self.messages.append({"role": "user", "content": "There is an error in the code provided. Please fix it."})
                self.messages.append({"role": "assistant", "content": test_code_output["stderr"]})
            else:
                try:
                    results = evaluate_titanic_submission(self.val_path, self.submission_val_path)
                    acc = results[2]['accuracy']
                    # Send feedback
                    if acc < 0.75:
                        self.messages.append({"role": "user", "content": "The accuracy is below the threshold of 0.75. Can you improve the code?"})
                        self.messages.append({"role": "assistant", "content": code})
                        improvement = 0
                    elif acc >= 0.75 and improvement >= 3:
                        # Stop looping
                        break
                    else:
                        # Ask for improvements
                        print("Asking for improvements")
                        improvement += 1
                        self.messages.append({"role": "user", "content": f"Can you improve the code to increase the accuracy? The current accuracy is {acc}. Use the old code from previous messages if the current accuracy is lower."})
                        self.messages.append({"role": "assistant", "content": code})
                except Exception as e:
                    # Fix missing files
                    self.messages.append({"role": "user", "content": "There are some files missing. Please fix it."})
                    self.messages.append({"role": "assistant", "content": e})

        if os.path.isfile(self.submission_val_path) and os.path.isfile(self.submission_test_path):
            print("Required files generated")
        else:
            print("Required files not generated")
        if acc is not None:
            print("Final accuracy: ", acc)
    ################################
    # Your code ends here
    ################################
